# Directional Consensus (DC): stress-testing a gold-free faithfulness score for NL→FOL

This notebook is a small, runnable version of the `method.py` entry point of the experiment
*"Stress-testing peer-agreement scores for logic translations"*.

**Where it fits.** NL→FOL outputs are hard to evaluate: a correct formula is not unique, and gold formulas are rare.
**Directional Consensus (DC)** needs no gold. Several systems formalise the same sentence (here 9 LLM systems).
For a candidate formula, DC is the **reliability-weighted share of the other systems' ("peer") formulas that are
logically EQUIVALENT to it**. Before the z3 equivalence check, the two formulas' predicates are aligned
without looking at their names:

* **L1**: a one-to-one mapping between the predicates of the two formulas,
* **L2**: a partial mapping,
* **L3**: granularity-aware name-free definitions (one predicate may correspond to a combination of predicates in the other formula).

$$\mathrm{DC}(f) = \frac{\sum_{p} w_p\,[\mathrm{rel}(f,p)=\mathrm{EQUIV}]}{\sum_{p:\,\mathrm{rel}\neq \mathrm{UNKNOWN}} w_p}$$

The weights $w_p$ are one-coin Dawid–Skene reliability estimates. If the candidate cannot be parsed, or fewer than 2 peers
are covered, DC = 0.5 and the item is marked `covered=False`.

**What `method.py` does.** It runs a chain of stages (prep → M0 … M7 → verdicts). Each stage is a separate script. The
expensive stages (about 74k z3 pair relations, LLM judge calls, 2,000-draw bootstraps) write caches to `work/`. The
last stage, `export()`, joins those caches into `method_out.json`, which has three datasets:

1. `heldout_confirm_dev`: 700 sentences × 9 systems. Each row has the candidate, a label (panel or solver),
   DC with its ladder variants, and the frozen baselines (LLM judge B1, sampling consensus LC_*, NLI/cosine round-trip B3, parse rate B7, …).
2. `m1_constructed_probes`: gold formulas and mutants of them (negation, quantifier swap, dropped or added condition, …), each in
   3 vocabulary arms: `conf` (original names), `tok` (random-token names) and `syn` (synonym names).
3. `m7_gold_as_peer`: the dataset's **gold** formula scored by DC against the 9 systems. This tests whether DC flags wrong gold annotations.

**What this demo runs.** The full pipeline needs the upstream datasets, z3 and about an hour of CPU. This notebook loads
`mini_demo_data.json` instead. It holds the cached inputs that `export()` reads (frame rows, `dc_scores`, probes, B1 on probes, gold scores),
restricted to **11 sentences × 9 systems = 99 candidate formalisations**, 100 M1 probes and 11 gold rows. The notebook then:
* runs the original `export()` code on them (only the input paths are changed),
* recomputes DC from the stored per-peer relations as a sanity check,
* computes AUROCs on the subset and plots them next to the full-scale results.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# numpy, scikit-learn, matplotlib — pre-installed on Colab, install locally only (Colab's exact versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

## Imports

The first block is the import block of `method.py`, copied unchanged. The second block adds what the notebook needs
for the AUROC tables and plots at the end.

In [ ]:
# --- original method.py imports ---
import json
import subprocess
import sys
from pathlib import Path

from loguru import logger

# --- extra imports for the demo analysis / plots ---
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

## Load the demo data

`mini_demo_data.json` is fetched from GitHub. If that fails, or the fetched file does not have this demo's keys, the notebook uses a local copy. The file has these keys:

| key | original source in the experiment workspace | used by |
|---|---|---|
| `G` | `work/frame.jsonl` rows (`fold == heldout_confirm`) joined by `m0_anchor.load_all()` with the frozen exp3 baseline scores (`r["m"]`) | dataset 1 & 3 |
| `sc` | `work/dc_scores.jsonl`: DC, ladder variants, per-peer relations, error type | dataset 1 |
| `m3` | `results/m3_boundary.json` (only `n_calls_arbiter`) | arbiter status |
| `m1_probes`, `m1_b1` | `work/m1_probes.jsonl`, `work/m1_b1.jsonl` | dataset 2 |
| `gold_dc_scores` | `work/gold_dc_scores.jsonl` | dataset 3 |
| `reference_full_m0` | `results/m0_anchor.json` (full-scale AUROCs, 609 panel / 5,847 solver items) | comparison plot |

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-abf543-do-sentence-and-formula-generalize-the/main/round-2/experiment-4/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            d = json.loads(response.read().decode())
            if "G" in d and "sc" in d: return d  # guard: ignore a stale/different file at the same URL
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["description"])
print({k: len(v) for k, v in data.items() if isinstance(v, (list, dict))})

## Configuration

`method.py` itself has no numeric hyper-parameters. The DC configuration was frozen before scoring
(`frozen_dc_config.json`, sha256 `7ebda7e2…`), and the heavy stages ran in separate scripts. The settings below
only choose **how much of the bundled data** to use and **how many bootstrap draws** the AUROC confidence intervals get.

The original experiment used all 700 DEV sentences (6,300 rows), all 223 probe sentences (3,175 probes) and `N_BOOT = 2000`
sentence-level bootstrap draws. The bundled file contains only 11 sentences, so `N_SENTENCES = 11` is the most this demo can use.

In [ ]:
N_SENTENCES = 11     # sentences (x 9 systems) taken from data["G"]; max 11 in the bundled file (original: 700)
N_BOOT = 2000        # sentence-bootstrap draws for AUROC CIs in the analysis section (original: 2000)
SEED = 0             # bootstrap RNG seed

## Logging and the original stage list

This setup is copied from `method.py`. One change: the file sink (`ROOT / "logs" / "method.log"`) is removed, because a notebook
has no `logs/` directory. `STAGES` is kept as documentation of the full pipeline. The scripts it lists are not part of this demo, so
`main("all")` is not run. Only the final `export()` stage runs, on the bundled caches.

In [ ]:
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
# (original also logged to ROOT / "logs" / "method.log" — dropped in the notebook)
ROOT = Path(".").resolve()
PY = sys.executable
STAGES = [["scripts/prep.py"], ["scripts/m6_record.py"], ["scripts/m0_pairs.py", "all"], ["scripts/m0_score.py", "all"],
          ["scripts/m0_anchor.py"], ["scripts/prereg.py"], ["scripts/m1_build_score.py", "all"], ["scripts/m1_b1.py"],
          ["scripts/m1_analyze.py"], ["scripts/m2_invariance.py", "1000"], ["scripts/m2_diagnose.py"], ["scripts/m3_boundary.py"],
          ["scripts/m3_analyze.py"], ["scripts/m4_placebo.py"], ["scripts/m5_m6_m7.py"], ["scripts/extra_analyses.py"],
          ["audit_rederive.py"], ["scripts/verdicts.py"]]

## Stand-ins for the cached inputs

In `method.py`, `export()` imports `load_all()` from `scripts/m0_anchor.py` and reads JSONL caches with `rj(WORK / …)`.
The functions below return the same objects from `data`:

* `load_all()` returns `G`, the dev rows (each with its baseline scores in `r["m"]`), and `sc`, the DC score records keyed by `item_id`.
  It is limited to the first `N_SENTENCES` sentences.
* `rj(name)` returns the rows of one bundled cache (`m1_b1.jsonl`, `m1_probes.jsonl` or `gold_dc_scores.jsonl`), limited to the same sentences.

`s()` is the original helper that formats a score as a 6-decimal string, or `""` for missing.

In [ ]:
_SIDS = list(dict.fromkeys(r["sentence_id"] for r in data["G"]))[:N_SENTENCES]
_SSET = set(_SIDS)
_CACHE = {"m1_b1.jsonl": data["m1_b1"], "m1_probes.jsonl": data["m1_probes"], "gold_dc_scores.jsonl": data["gold_dc_scores"]}


def load_all():
    G = [dict(r) for r in data["G"] if r["sentence_id"] in _SSET]
    sc = {k: v for k, v in data["sc"].items() if v["sid"] in _SSET}
    return G, sc


def rj(name):
    return [x for x in _CACHE[name] if x["sid"] in _SSET]


def s(x):
    return "" if x is None else (f"{x:.6f}" if isinstance(x, float) else str(x))


print(f"using {len(_SIDS)} sentences:", _SIDS)

## `export()`: build the three output datasets

This is the `export()` function of `method.py`, copied as closely as possible. The only changes are the data sources:
`load_all()`/`rj()` read from `data` (see the previous cell), `m3` is `data["m3"]`, and the frozen-config hash is read
from `data` instead of `frozen_dc_config.sha256`.

**Dataset 1, `heldout_confirm_dev`.** There is one example per (sentence, system) candidate. `output` is the label
(`faithful`/`unfaithful`/`unknown`). `predict_*` holds DC and its ladder variants:
* `S0_L1`, `S1_L2`, `S2_L3`: unweighted DC using alignment up to L1, L2 or L3,
* `DC_L2w`: weighted DC without L3,
* `DC_lex`: DC with name-based (lexical) alignment,
* `LC_maj_star`: majority-cluster membership,
* `VC`: vocabulary conformity, a control score.

`predict_baseline_*` holds the frozen baselines from the previous round. `DC_arb` is set equal to DC, because the text-anchored arbiter
was **not run** (OpenRouter budget 403). The metadata keeps the per-peer relations, the strength profile
(stronger/weaker/contradictory mass) and the error type implied by the relation to the majority cluster.

**Dataset 2, `m1_constructed_probes`.** Each gold formula (`kind=F`) and each mutant (`kind=M`, with the mutation operator in `op`)
is scored by DC. B1 is the LLM judge. The probes come in three vocabulary arms: original names, random-token names and synonym names.

**Dataset 3, `m7_gold_as_peer`.** For each sentence, DC scores the dataset's gold formula against the 9 systems. `output` says
whether the gold was judged faithful in the audit.

In [ ]:
@logger.catch(reraise=True)
def export():
    # original: from dc.common import WORK, RES, EXP3, load_frame, rj ; from m0_anchor import load_all
    G, sc = load_all()
    arb = {}
    m3 = data["m3"]  # original: json.loads((RES / "m3_boundary.json").read_text())
    arb_status = "RUN" if m3.get("n_calls_arbiter") else "NOT_RUN: OpenRouter 403 aii_run_budget_exhausted (DC_arb = DC)"
    ex1 = []
    for r in G:
        x = sc[r["item_id"]]
        e = {"input": json.dumps({"sentence": r["sentence"], "candidate_fol": r["candidate_fol"]}, ensure_ascii=False),
             "output": r["output"]}
        for m in ("DC", "DC_L2w", "DC_lex", "S0_L1", "S1_L2", "S2_L3", "LC_maj_star", "VC"):
            e[f"predict_{m}"] = s(x.get(m))
        e["predict_DC_arb"] = s(x.get("DC"))
        for m in ("B1", "LC_maj", "LC_onecoin", "LC_ds_binary", "B3nli", "B3cos", "B7", "B8"):
            if r["m"].get(m) is not None:
                e[f"predict_baseline_{m}"] = s(r["m"][m])
        e.update({"metadata_fold": "heldout_confirm_dev", "metadata_item_id": r["item_id"], "metadata_sentence_id": r["sentence_id"],
                  "metadata_system": r["system"], "metadata_corpus": r["corpus"], "metadata_complexity_tercile": r["complexity_tercile"],
                  "metadata_n_conditions": r["n_conditions"], "metadata_label_source": r["label_source"],
                  "metadata_L3_sampling_weight": r["L3_sampling_weight"], "metadata_L3_primary_error": r["L3_primary_error"],
                  "metadata_DC_covered": x.get("DC_cov"), "metadata_error_type": x.get("error_type"),
                  "metadata_rel_to_mode": x.get("rel_to_mode"), "metadata_in_mode": x.get("in_mode"),
                  "metadata_strength_profile": x.get("strength_profile"), "metadata_coverage": x.get("coverage"),
                  "metadata_peer_relations": {p["peer"]: p["rel"] for p in x.get("per_peer", [])},
                  "metadata_seconds": x.get("seconds"), "metadata_usd": 0.0, "metadata_arbiter_status": arb_status})
        ex1.append(e)
    b1 = {(q["sid"], q["arm"], q["kind"], q["op"]): q for q in rj("m1_b1.jsonl")}
    ex2 = []
    for p in rj("m1_probes.jsonl"):
        e = {"input": json.dumps({"sentence": p["sentence"], "candidate_fol": p["fol"]}, ensure_ascii=False),
             "output": "faithful" if p["kind"] == "F" else "unfaithful"}
        for m in ("DC", "DC_L2w", "S0_L1", "S2_L3", "LC_maj_star", "VC"):
            e[f"predict_{m}"] = s(p.get(m))
        q = b1.get((p["sid"], p["arm"], p["kind"], p["op"]))
        if q:
            e["predict_baseline_B1"] = s(q["B1"])
        e.update({"metadata_fold": "m1_probe", "metadata_sentence_id": p["sid"], "metadata_arm": p["arm"],
                  "metadata_kind": p["kind"], "metadata_operator": p["op"], "metadata_iso_L1": p.get("iso_L1"),
                  "metadata_fresh_pred": p.get("fresh_pred"), "metadata_type_contract": p.get("type_contract"),
                  "metadata_type_repair": p.get("type_repair"), "metadata_corpus": p.get("corpus"),
                  "metadata_peer_relations": p.get("rels")})
        ex2.append(e)
    gd = {x["item_id"]: x for x in rj("gold_dc_scores.jsonl")}
    ex3 = []
    seen = set()
    for r in G:
        if r["sentence_id"] in seen:
            continue
        seen.add(r["sentence_id"])
        g = gd.get(f"{r['sentence_id']}:GOLD")
        if g is None or r.get("gold_faithful_final") is None:
            continue
        ex3.append({"input": json.dumps({"sentence": r["sentence"], "candidate_fol": r["gold_fol_original"]}, ensure_ascii=False),
                    "output": "faithful" if r["gold_faithful_final"] else "unfaithful",
                    "predict_DC_gold_as_peer": s(g["DC"]), "predict_S0_L1_gold": s(g["S0_L1"]),
                    "metadata_fold": "m7_gold", "metadata_sentence_id": r["sentence_id"], "metadata_corpus": r["corpus"],
                    "metadata_DC_covered": g.get("DC_cov"), "metadata_error_type_vs_mode": g.get("error_type"),
                    "metadata_gold_audit_primary_error": r.get("gold_audit_primary_error")})
    out = {"metadata": {"method_name": "Directional Consensus (DC)",
                        "description": "Gold-free NL->FOL faithfulness: reliability-weighted share of peer formalisations "
                                       "logically EQUIVALENT to the candidate up to a lexical-free (granularity-aware) "
                                       "alignment; stress-tested on constructed truth (DEV set, not confirmatory)",
                        "frozen_dc_config_sha256": data["frozen_dc_config_sha256"],  # original: (ROOT / "frozen_dc_config.sha256").read_text().strip()
                        "labels": "output = dataset label (panel3 for 609 rows, else audited-solver; unknown kept)",
                        "arbiter_status": arb_status,
                        "predictions": "predict_* are scores in [0,1] as strings; predict_baseline_* are frozen exp3 baselines"},
           "datasets": [{"dataset": "heldout_confirm_dev", "examples": ex1}, {"dataset": "m1_constructed_probes", "examples": ex2},
                        {"dataset": "m7_gold_as_peer", "examples": ex3}]}
    (ROOT / "method_out.json").write_text(json.dumps(out, ensure_ascii=False))
    logger.info(f"method_out.json: {len(ex1)} + {len(ex2)} + {len(ex3)} examples")
    return out

## Run the export stage

`method.py export` ran only this function. `method.py all` ran every script in `STAGES` first and then this function.
The call below also writes `method_out.json` next to the notebook.

In [ ]:
out = export()
ds = {d["dataset"]: d["examples"] for d in out["datasets"]}
print(json.dumps(out["metadata"], indent=1))
ex = ds["heldout_confirm_dev"][0]
print("\nfirst dev example:")
print(json.dumps({k: v for k, v in ex.items() if not k.startswith("metadata_") or k in
                  ("metadata_system", "metadata_rel_to_mode", "metadata_error_type", "metadata_peer_relations")},
                 indent=1, ensure_ascii=False))

## Sanity check: recompute DC from the per-peer relations

`dc_scores` stores, for each candidate, the relation to every peer (`EQUIV`, `STRONGER`, `WEAKER`, `COMPATIBLE`,
`CONTRADICTORY`, `UNALIGNABLE`, `UNKNOWN`, `PEER_UNPARSEABLE`) and that peer's Dawid–Skene weight. The cell below applies the
formula from `dc.core.directional_consensus`:

`DC = Σ w·[rel = EQUIV] / Σ_{rel ∉ {UNKNOWN, *_UNPARSEABLE}} w`, with DC = 0.5 when fewer than 2 peers are covered.

The recomputed values should match the stored DC scores.

In [ ]:
REL_OK = ("EQUIV", "STRONGER", "WEAKER", "COMPATIBLE", "CONTRADICTORY")
G_used, sc_used = load_all()
diffs = []
for r in G_used:
    x = sc_used[r["item_id"]]
    num = den = 0.0
    for p in x["per_peer"]:
        if p["rel"] in ("PEER_UNPARSEABLE", "SELF_UNPARSEABLE", "UNKNOWN"):
            continue
        den += p["w"]
        num += p["w"] * (p["rel"] == "EQUIV")
    n_cov = sum(p["rel"] in REL_OK for p in x["per_peer"])
    dc = num / den if (n_cov >= 2 and den > 0) else 0.5
    diffs.append(abs(dc - x["DC"]))
print(f"recomputed DC for {len(diffs)} candidates; max |recomputed - stored| = {max(diffs):.2e}")

## Results and visualisation

**(a) Item-level AUROC.** How well does each score separate faithful from unfaithful candidates? This is computed on the demo
subset, with a sentence-level bootstrap CI (`N_BOOT` draws), and shown next to the full-scale DEV numbers from
`results/m0_anchor.json`. The full-scale AUROCs use 609 panel-labelled items (weighted by `L3_sampling_weight`) and 5,847
solver-labelled items. With only 11 sentences the demo CIs are wide. The comparison shows that the pipeline computes the numbers
correctly; the demo subset is too small to reproduce the full-scale results.

In [ ]:
METRICS = ["DC", "DC_L2w", "S0_L1", "S1_L2", "S2_L3", "DC_lex", "LC_maj_star", "VC",
           "baseline_LC_maj", "baseline_LC_ds_binary", "baseline_B1", "baseline_B3nli", "baseline_B7"]
rows = [e for e in ds["heldout_confirm_dev"] if e["output"] in ("faithful", "unfaithful")]
rng = np.random.default_rng(SEED)
sids = sorted({e["metadata_sentence_id"] for e in rows})
by_sid = defaultdict(list)
for e in rows:
    by_sid[e["metadata_sentence_id"]].append(e)


def auroc(ex, m):
    y = [1 if e["output"] == "faithful" else 0 for e in ex if e.get(f"predict_{m}", "") != ""]
    p = [float(e[f"predict_{m}"]) for e in ex if e.get(f"predict_{m}", "") != ""]
    return roc_auc_score(y, p) if len(set(y)) == 2 else np.nan


table = {}
for m in METRICS:
    a = auroc(rows, m)
    boots = []
    for _ in range(N_BOOT):
        bs = rng.choice(sids, size=len(sids), replace=True)
        boots.append(auroc([e for s_ in bs for e in by_sid[s_]], m))
    boots = np.array(boots, dtype=float)
    lo, hi = (np.nanpercentile(boots, [2.5, 97.5]) if np.isfinite(boots).any() else (np.nan, np.nan))
    n = sum(e.get(f"predict_{m}", "") != "" for e in rows)
    ref = data["reference_full_m0"]["solver_unweighted"].get(m.replace("baseline_", ""), {})
    refp = data["reference_full_m0"]["panel_weighted"].get(m.replace("baseline_", ""), {})
    table[m] = (a, lo, hi, n, ref.get("auroc"), refp.get("auroc"))

print(f"labelled demo items: {len(rows)} from {len(sids)} sentences "
      f"({sum(e['output'] == 'faithful' for e in rows)} faithful / {sum(e['output'] == 'unfaithful' for e in rows)} unfaithful)")
print(f"{'metric':<24}{'demo AUROC':>11}{'95% CI':>18}{'n':>5}{'full solver':>13}{'full panel':>12}")
for m, (a, lo, hi, n, ref, refp) in table.items():
    f = lambda v: "   -  " if v is None or not np.isfinite(v) else f"{v:.3f}"
    print(f"{m:<24}{f(a):>11}   [{f(lo)}, {f(hi)}]{n:>5}{f(ref):>13}{f(refp):>12}")

**(b) M1 constructed probes (vocabulary × meaning).** Mean score of faithful gold formulas (F) and mutants (M) in each vocabulary
arm. Every probe's peers are the 9 systems' real formulas for that sentence. On the full data DC is almost exactly
vocabulary-invariant (gap 0.0015). The LLM judge B1 drops from AUROC 0.945 to 0.794 when predicate names are random tokens.

**(c) M7 gold-as-peer.** DC scores each dataset gold formula against the 9 systems. On the full data, a low DC flags audited-wrong gold
with AUROC 0.837.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

# (a) AUROC demo vs full
ms = [m for m in METRICS if np.isfinite(table[m][0])]
x = np.arange(len(ms))
a = np.array([table[m][0] for m in ms]); lo = np.array([table[m][1] for m in ms]); hi = np.array([table[m][2] for m in ms])
full = np.array([table[m][4] if table[m][4] is not None else np.nan for m in ms], dtype=float)
ax = axes[0]
ax.bar(x - 0.2, a, 0.4, yerr=[np.clip(a - lo, 0, None), np.clip(hi - a, 0, None)], capsize=2,
       label=f"demo subset ({len(sids)} sent.)", color="#4C72B0")
ax.bar(x + 0.2, full, 0.4, label="full DEV (solver labels)", color="#DD8452")
ax.axhline(0.5, color="grey", lw=0.8, ls="--")
ax.set_xticks(x, [m.replace("baseline_", "b:") for m in ms], rotation=60, ha="right", fontsize=8)
ax.set_ylim(0, 1.18); ax.set_ylabel("AUROC (faithful vs unfaithful)"); ax.set_title("(a) item-level AUROC")
ax.legend(fontsize=8, loc="upper center", ncol=2)

# (b) M1 probes: mean DC / B1 for F vs M per vocabulary arm
pr = ds["m1_constructed_probes"]
arms = ["conf", "tok", "syn"]
ax = axes[1]
w = 0.2
for j, (metric, kind, col) in enumerate([("DC", "F", "#4C72B0"), ("DC", "M", "#A6BDDB"),
                                         ("baseline_B1", "F", "#C44E52"), ("baseline_B1", "M", "#F4A582")]):
    vals = []
    for arm in arms:
        v = [float(e[f"predict_{metric}"]) for e in pr if e["metadata_arm"] == arm and e["metadata_kind"] == kind
             and e.get(f"predict_{metric}", "") != ""]
        vals.append(np.mean(v) if v else np.nan)
    ax.bar(np.arange(3) + (j - 1.5) * w, vals, w, color=col, label=f"{metric.replace('baseline_', '')} · {'faithful' if kind == 'F' else 'mutant'}")
ax.set_xticks(range(3), ["conf\n(original names)", "tok\n(random tokens)", "syn\n(synonyms)"])
ax.set_ylim(0, 1.05); ax.set_ylabel("mean score"); ax.set_title(f"(b) M1 probes (n={len(pr)})")
ax.legend(fontsize=8)

# (c) M7: DC of the gold formula, split by audited gold faithfulness
g7 = ds["m7_gold_as_peer"]
ax = axes[2]
for k, (lab, col) in enumerate([("faithful", "#55A868"), ("unfaithful", "#C44E52")]):
    v = [float(e["predict_DC_gold_as_peer"]) for e in g7 if e["output"] == lab]
    ax.scatter(np.full(len(v), k) + rng.uniform(-0.08, 0.08, len(v)), v, color=col, s=45, zorder=3)
    if v:
        ax.hlines(np.mean(v), k - 0.25, k + 0.25, color="k", lw=2)
ax.set_xticks([0, 1], ["gold audited\nfaithful", "gold audited\nWRONG"])
ax.set_ylim(-0.05, 1.05); ax.set_ylabel("DC(gold) vs 9 systems"); ax.set_title(f"(c) M7 gold-as-peer (n={len(g7)})")
plt.tight_layout(); plt.show()

**(d) Error typing and per-peer relations.** For every *unfaithful* candidate in the demo subset, the table below shows the
DC score, the candidate's relation to the majority ("mode") cluster of the systems, the error type DC infers from that relation,
and the panel's primary error label when one exists.

In [ ]:
print(f"{'system':<22}{'DC':>6}  {'rel_to_mode':<14}{'DC error_type':<30}{'panel error':<26}candidate")
for e in ds["heldout_confirm_dev"]:
    if e["output"] != "unfaithful":
        continue
    cand = json.loads(e["input"])["candidate_fol"]
    print(f"{e['metadata_system']:<22}{float(e['predict_DC']):>6.2f}  {str(e['metadata_rel_to_mode']):<14}"
          f"{str(e['metadata_error_type']):<30}{str(e['metadata_L3_primary_error']):<26}{cand[:70]}")

rel_counts = Counter(r for e in ds["heldout_confirm_dev"] for r in e["metadata_peer_relations"].values())
print("\nper-peer relation counts over all demo candidates:", dict(rel_counts.most_common()))

### Takeaways from the full experiment

These numbers are from the full-scale run. The demo subset above shows the same pipeline on 99 items.

* **M0 (anchor).** DC panel AUROC is 0.769 [0.717, 0.819] (solver labels: 0.815). This is about the same as simple majority consensus
  LC_maj (0.765), LC_ds (0.787) and the LLM judge B1 (0.762). In stacking, DC adds nothing over LC_maj (+0.015 [−0.009, 0.040]).
* **M1 (vocabulary × meaning).** DC is exactly vocabulary-invariant. The LLM judge is not: B1 drops from 0.945 to 0.794 with random tokens.
  **However**, the L3 name-free definitions *absorb* negation (NEG AUROC 0.51 vs 1.00 without L3) and dropped restrictors (0.66 vs 0.99).
  For this reason `DC_L2w` (no L3) is the proposed amendment.
* **M5 (ladder).** L2 partial alignment is the whole mechanism (+0.128). L3 and Dawid–Skene weighting are neutral.
* **M3 (shared bias).** DC can only be as right as the peer majority. When wrong clusters dominate, within-sentence concordance falls to 0.45.
* **M7 (gold as peer).** DC(gold) flags wrong gold with AUROC 0.837 [0.809, 0.864]. This adds +0.070 over B1.
* Pre-registered verdicts: 9 PASS / 10 FAIL / 1 NOT_RUN (the text-anchored arbiter was not run because of an API budget 403).